## Scikit-learn para realizar selección de variables

- [Análisis de correlaciones](#Análisis-de-correlaciones)
- [Métodos basados en filtros](#Métodos-basados-en-filtros)
- [Métodos basados en wrappers](#Métodos-basados-en-wrappers)
- [PCA: Principal component analysis](#PCA:-Principal-component-analysis)

## Análisis de correlaciones

### Visualización de la matriz de correlaciones

Para obtener y mostrar visualmente la matriz de correlaciones entre todas las variables de entrada se puede utilizar la librería Pandas y, en concreto, la función [*corr*](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.corr.html):

```
    import matplotlib.pyplot as plt

    # Cálculo de la matriz de correlaciones con la función corr de pandas
    correlaciones = X.corr()

    # Mostramos la matriz de correlaciones especificando el rango de los valores [-1, 1]
    fig = plt.figure(figsize=(50,50))
    ax = fig.add_subplot(111)
    cax = ax.matshow(correlaciones, vmin=-1, vmax=1, cmap=plt.cm.rainbow)
    fig.colorbar(cax)
    ticks = np.arange(0,len(X_train.columns),1)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    # Añadimos los nombres de las variables en la figura
    names = X.columns
    ax.set_xticklabels(names, rotation='90')
    ax.set_yticklabels(names)
    plt.show()
```

La matriz de correlaciones es simétrica (matriz triangular superior e inferior iguales). Además, la correlación entre una variable y ella misma es perfecta (diagonal con unos: rojo).

Podemos crear una clase para seleccionar las variables que no estén correlacionadas entre ellas de esta forma:

```
    from sklearn.base import TransformerMixin

class corr_selection(TransformerMixin):

        # Constructor de la clase
        def __init__(self, umbral=0.9, verbose=False):
            # Umbral deseado para determinar variables correlacionadas
            self.umbral = umbral
            # Parámetro que determina si imprimir información del proceso o no
            self.verbose = verbose

        # Método fit
        def fit(self, X, y=None):
            # Transformamos X en DataFrame por si acaso llega en formato ndarray
            X = pd.DataFrame(X)
            # Calculamos la matriz de correlaciones con la función corr de pandas sobre el DataFrame con las variables de entrada X y la ponemos en valor absoluto
            correlaciones = X.corr().abs()
            # Seleccionamos el triángulo superior de la matriz de correlación
            upper = correlaciones.where(np.triu(np.ones(correlaciones.shape), k=1).astype('bool'))
            # Obtenemos los nombres de aquellas variables con correlación mayor al umbral deseado
            self.variables_a_eliminar = list(set([column for i,column in enumerate(upper.columns) if any(upper[column] > self.umbral)]))

             # Si queremos mostrar información se muestra el número de variables eliminadas y sus nombres
            if self.verbose:
                print('Se han eliminado {} variables, que son: {}'.format(len(self.variables_a_eliminar), self.variables_a_eliminar))
            # Devolvemos el objeto modificado (en este caso ha aprendido qué variables se deben eliminar al estar correlacionadas con otras)
            return self

        # Método transform
        def transform(self, X):
            # Transformamos X en DataFrame por si acaso llega en formato ndarray
            X = pd.DataFrame(X)
            # Creamos una copia del DataFrame X para no perder los datos originales
            X_uncorr = X.copy()
            # Eliminamos las variables con alta correlación con algunda de las variables de entrada
            X_uncorr.drop(self.variables_a_eliminar, axis=1, inplace=True)
            # Devolvemos el DataFrame transformado
            return X_uncorr

        # Método para asignar los valores de los híper-parámetros y que, de este modo, 
            # podamos aplicar GridSearchCV sobre un objeto de esta clase
        def set_params(self, **parameters):
            for parameter, value in parameters.items():
                setattr(self, parameter, value)
            return self

        # Método para obtener los valores de los híper-parámetros que queramos del modelo (lo usa GridSearchCV al mostrar la mejor configuración)
        def get_params(self, deep=True):
            # Devolvemos los valores de los híper-parámetros del método de preparación de datos
            return {"umbral": self.umbral}
```

### Métodos basados en filtros

La librería scikit-learn nos ofrece una librería para realizar la selección variables. Esta librería se llama [*feature_selection*](http://scikit-learn.org/stable/modules/feature_selection.html).

    from sklearn import feature_selection
    
En primer lugar vamos a mencionar las técnicas basadas en filtros uni-variable. Los filtros uni-variables aplican una medida (habitualmente estadística) que determina la calidad de las variables individuales. Scikit-learn provee tres métricas de calidad de las variables para resolver **problemas de clasificación**:
* Chi cuadrado: función [*chi2*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.chi2.html#sklearn.feature_selection.chi2)
* ANOVA: función [*f_classif*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html#sklearn.feature_selection.f_classif)
* Información mutua: función [*mutual_info_classif*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html#sklearn.feature_selection.mutual_info_classif)

Para resolver **problemas de regresión** ofrece:
* Correlación mutua: función [*f_regression*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_regression.html#sklearn.feature_selection.f_regression)
* Información mutua: función [*mutual_info_regression*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_regression.html#sklearn.feature_selection.mutual_info_regression)

Una vez conocida la calidad de cada variable se deben escoger las mejores. Para ello vimos en las clases teóricas que había varias opciones. La librería Scikit-learn ofrece dos de estas técnicas en forma de clases (con sus campos y sus métodos):
* Elegir las k mejores con la clase [*SelectKBest*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html#sklearn.feature_selection.SelectKBest)
    
    SelectKBest(score_func=funcionCalidad, k=valorK)
    
    
* Elegir las variables en base al percentil (el % de las variables) con la clase [*SelectPercentile*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectPercentile.html#sklearn.feature_selection.SelectPercentile)
    
    SelectPercentile(score_func=funcionCalidad, percentile=porcentajeVariables)

En ambas clases, elegir las k mejores o las que estén en el percentil deseado, en la llamada al constructor se debe especificar:
* La medida de calidad (funcionCalidad) de las variables. Es decir, cualquiera de las 3 funciones comentadas anteriormente
    * chi2
    * f_classif
    * mutual_info_classif
* El valor del parámetro k (valorK) para el método de las k mejores o el valor del percentil (porcentajeVariables) para el método de selección en base al percentil.

En ambas clases disponemos de los mismos métodos para realizar la selección de las variables:
* fit: Recibe como parámetro de entrada los datos de entrada (X) y de salida (Y). Esta función aplica la medida de calidad pasada como parámetro de entrada (funcionCalidad) sobre cada variable y establece la importancia de cada una de ellas.
* transform: Recibe como parámetro de entrada los datos de entrada (X). Esta función devuelve los datos de entrada con la selección de variables realizada (X'). Es decir, X' tendrá el mismo número de ejemplos pero menor número de variables (solamente las seleccionadas).
* fit_transform: Recibe como parámetro de entrada los datos de entrada (X) y de salida (Y). Esta función aplica secuencialmente las dos funciones anteriores.
* get_support: esta función devuelve un array de booleanos con tantos elementos como variables. En cada posición tendrá True si la variable asociada ha sido seleccionada y False en caso contrario.
* inverse_transform: Recibe como parámetro de entrada los datos de entrada con la selección de variables realizada (X'). Esta función deshace la selección de variables, es decir, devuelve el conjunto de datos original (X).

### Métodos basados en wrappers

La librería Scikit-learn también nos ofrece varias clases para poder implementar Wrappers. Todas ellas están dentro de la librería *feature_selection*.

La primera de ellas se llama [*RFE*](http://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html#sklearn.feature_selection.RFE) y corresponde al método Sequential Backward Selecion (SBS) explicado en clase de teoría. Es decir, es el método en el que se comienza con todas las variables del problema e iterativamente se va eliminando la peor variable de las que queden. En el caso de la implementación de esta clase lo que se hace es lo siguiente:
* Se aprende un clasificador (pasado como parámetro de entrada: clasificador) con todas las variables y se asignan pesos a cada una de las variables.
* Se elimina(n) variable(s) (el número de variables a eliminar se pasa como parámetro de entrada: numeroVariablesEliminarEnPaso) cuyos pesos en valor absoluto sean los menores.
* Luego se vuelve a entrenar con las variables que queden y se vuelven a calcular los pesos de dichas variables.
* Este proceso se repite hasta que se alcance el número de variables a mantener (pasado como parámetro de entrada: numeroVariablesMantener)

La llamada al constructor de la clase es la siguiente:

    RFE(estimator, n_features_to_select=numeroVariablesMantener, step=numeroVariablesEliminarEnPaso)
    
Los híper-parámetros de esta clase son los siguientes:
* estimator: método de aprendizaje a utilizar en el *wrapper*.
* numeroVariablesEliminarEnPaso: número de variables a eliminar en cada iteración del proceso (elima las variables cuyos pesos, en valor absoluto, sean los menores).
* numeroVariablesMantener: número de variables a seleccionar.
    
Esta clase tiene varios métodos, entre ellos los mismos que se han explicado para los filtros uni-variables utilizados en la primera parte de la práctica:
* fit: Recibe como parámetro de entrada los datos de entrada (X) y de salida (Y). Esta función aplica la medida de calidad pasada como parámetro de entrada (funcionCalidad) sobre cada variable y establece la importancia de cada una de ellas.
* transform: Recibe como parámetro de entrada los datos de entrada (X). Esta función devuelve los datos de entrada con la selección de variables realizada (X'). Es decir, X' tendrá el mismo número de ejemplos pero menor número de variables (solamente las seleccionadas).
* fit_transform: Recibe como parámetro de entrada los datos de entrada (X) y de salida (Y). Esta función aplica secuencialmente las dos funciones anteriores.
* get_support: esta función devuelve un array de booleanos con tantos elementos como variables. En cada posición tendrá True si la variable asociada ha sido seleccionada y False en caso contrario.
* inverse_transform: Recibe como parámetro de entrada los datos de entrada con la selección de variables realizada (X'). Esta función deshace la selección de variables, es decir, devuelve el conjunto de datos original (X).

NOTA: el clasificador utilizado en el wrapper (evaluación de la importancia de las variables) tiene que actualizar el atributo *coef_* de su clase correspondiente, como, por ejemplo, la regresión logística (*feature_selection.RFE(linear_model.LogisticRegression(solver='liblinear')*). En caso de que no lo haga no se puede utilizar como clasificador para la eliminación secuencial de variables.

También está disponible la clase [*SequentialFeatureSelector*](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) que permite realizar la introducción escalonada de la mejor variable (híper-parámetro *direction='forward'*) o la eliminación escalonada de la peor variable (*direction='backward'*). Además, en este caso la calidad de las variables se determina mediante un proceso de validación cruzada (híper-parámetro *cv*, cuyo valor por defecto es 5). Además, se pueden determinar el número de *cores* del ordenador a utilizar con el parámetro llamado *n_jobs* (-1 indica que se usen todos los disponibles). El resto de híper-parámetros son los mismos que en la clase anterior.

Las clases anteriores tienen como característica que debemos seleccionar el número de variables a seleccionar. Sin embargo, puede que el número que asignemos no sea la mejor opción. Para abordar este problema, Scikit-learn ofrece la clase [*RFECV*](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html#sklearn.feature_selection.RFECV) que aplica la eliminación de la peor variable en cada iteración (RFE) y además obtiene el mejor número de variables a seleccionar aplicando una validación cruzada (de ahí el CV del nombre de la clase, por defecto aplica la validacion cruzada de 5 particiones).

Si se desea mostrar el número de variables seleccionadas se puede consultar el parámetro *n_features_* del objeto de *RFECV*. En caso de que este componente esté dentro de una Pipeline se puede acceder con la propiedad *named_steps* de la clase *Pipeline* (y luego mirando la propiedad mencionada anteriormente). 

### PCA: Principal component analysis

Una técnica muy habitual a la hora de realizar reducción de datos por medio de transformaciones de variables es el análisis de las componentes principales (PCA, de sus siglas en inglés). Scikit-learn nos ofrece dicha técnica implementada en la clase [*PCA*](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) de la librería *decomposition*. El constructor con los parámetros que vamos a utilizar en esta práctica es el siguiente:
    
    sklearn.decomposition.PCA(n_components: numCom, svd_solver='full')
    
El parámetro n_components puede tener diferentes valores al utilizar *svd_solver='full'*:
* Si *numCom* es un valor entero especifica el númerto de componentes principales a mantener.
* Si *numCom* es un valor real entre 0 y 1 especifica el porcentaje de la varianza que representan los  componentes principales seleccionados. 

El porcentaje de varianza que representa cada componente principal se puede encontrar en el atributo *explained_variance_ratio_* de la clase PCA. El número de componentes principales seleccionados se puede obtener en el atributo *n_components_* de la clase PCA.

Al igual que el resto de técnicas de preprocesamiento, para ejecutar PCA se debe llamar al constructor de la clase PCA, luego entrenarlo, método *fit*, con los ejemplos de entrenamiento y finalmente el objeto entrenado se puede utilizar para transformar ejemplos a la nueva base mediante el método *transform*.

Algo interesante de PCA es que podemos analizar la influencia del número de componentes principales utilizados. Scikit-learn nos devuelve los componentes principales ordenados por importancia (varianza de información que representan). Por este motivo, el primer componente principal será el más importante, luego el segundo, etc... y se puede analizar el resultado de PCA si sólo utilizamos el PC más importante, los dos más importantes, etc...